# Data Validation & Cleaning

Before any analysis or modeling, it is critical to ensure that the data used in this project
is reliable, consistent, and aligned with business rules.

In this notebook, we:
- Load all source datasets
- Validate key relationships
- Identify and correct data quality issues
- Prepare clean datasets for downstream analysis

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
plt.style.use("default")

## Load Datasets

The project uses four primary datasets:
- Sales transactions
- Inventory snapshots
- Product master
- Supplier master


In [2]:
import sys
sys.path.append('../')

# Load raw data
from src.utils import load_raw_data

sales_df, inventory_df, products_df, suppliers_df = load_raw_data()

print("Sales shape:", sales_df.shape)
print("Inventory shape:", inventory_df.shape)
print("Products shape:", products_df.shape)
print("Suppliers shape:", suppliers_df.shape)


Sales shape: (14640, 26)
Inventory shape: (40, 7)
Products shape: (40, 7)
Suppliers shape: (10, 4)


## Initial Data Overview

We first inspect the structure, column names, and data types of each dataset
to ensure they match the definitions provided in the data dictionary.


In [3]:
sales_df.head()

,date,sku_id,units_sold,selling_price,gross_revenue,promo_flag,discount_pct,month,week_of_year,day_of_week,is_weekend,is_festival_month,is_payday_period,season_tag,holiday_flag,weather_index,campaign_intensity,platform_traffic_source,traffic_index,competitor_price_index,competitor_stockout_flag,bundle_offer_flag,stock_visibility_score,rating_score,review_volume,product_visibility_rank
0,1/1/24,SKU0001,28,1028.54,28799.12,0,0.0072,1,1,0,0,0,1,winter,0,1.033050,1,affiliate,1.028220,1.039897,0,0,0.332970,4.99,220,14
1,1/2/24,SKU0001,27,942.76,25454.52,0,0.0900,1,1,1,0,0,1,winter,0,1.075864,0,organic,1.000124,1.034213,0,0,0.332574,4.99,220,26
2,1/3/24,SKU0001,27,1023.78,27642.06,0,0.0118,1,1,2,0,0,1,winter,0,0.776739,0,paid,1.042157,1.019800,0,0,0.369938,4.99,220,50
3,1/4/24,SKU0001,29,940.90,27286.10,0,0.0918,1,1,3,0,0,1,winter,0,1.029230,0,paid,0.998036,1.004657,0,0,0.445269,4.99,220,42
4,1/5/24,SKU0001,37,812.43,30059.91,1,0.2158,1,1,4,0,0,1,winter,0,1.093677,1,organic,1.047220,0.956571,0,0,0.414885,4.99,220,20


In [4]:
inventory_df.head()

,sku_id,current_stock,safety_stock,reorder_point,lead_time_days,inbound_shipment_qty,supplier_id
0,SKU0001,963,282,631,10,127,SUP003
1,SKU0002,581,213,428,6,36,SUP007
2,SKU0003,395,108,259,5,19,SUP006
3,SKU0004,375,113,274,5,40,SUP006
4,SKU0005,339,160,353,5,64,SUP008


In [5]:
products_df.head()

,sku_id,sku_name,category,sub_category,mrp,cost_price,supplier_id
0,SKU0001,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003
1,SKU0002,AirWave Earbuds,Beauty & Personal Care,Body Care,435,318.36,SUP007
2,SKU0003,GlowCare Serum,Home Cleaning,Multi-Surface,249,138.91,SUP006
3,SKU0004,FlexFit Yoga Mat,Electronics Accessories,Audio,532,290.52,SUP006
4,SKU0005,PureHome Dish Liquid,Electronics Accessories,Audio,654,477.19,SUP008


In [6]:
suppliers_df.head()

,supplier_id,supplier_name,avg_lead_time,lead_time_variability
0,SUP001,EastBridge Trading Co.,3,2
1,SUP002,NovaSupply Distributors,11,4
2,SUP003,Skyline Imports Pvt. Ltd.,10,2
3,SUP004,UrbanEdge Wholesale,3,4
4,SUP005,PrimeRoute Logistics,13,2


## Date Validation

Dates must be correctly parsed and sorted chronologically.
Incorrect date formats or unordered timestamps can break time-series analysis.

In [7]:
sales_df["date"] = pd.to_datetime(sales_df["date"])

sales_df = sales_df.sort_values(["sku_id", "date"])

C:\Users\Punk\AppData\Local\Temp\ipykernel_11228\966014289.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sales_df["date"] = pd.to_datetime(sales_df["date"])


## Key Consistency Checks

We validate whether all SKUs and suppliers referenced in transactional data
exist in the corresponding master datasets.

In [8]:
sales_skus = set(sales_df["sku_id"].unique())
inventory_skus = set(inventory_df["sku_id"].unique())
product_skus = set(products_df["sku_id"].unique())

missing_sales_skus = sales_skus - product_skus
missing_inventory_skus = inventory_skus - product_skus

print("SKUs in sales but missing in product master:", len(missing_sales_skus))
print("SKUs in inventory but missing in product master:", len(missing_inventory_skus))

SKUs in sales but missing in product master: 0
SKUs in inventory but missing in product master: 0


## Supplier Consistency Check

In [9]:
supplier_ids_products = set(products_df["supplier_id"].unique())
supplier_ids_master = set(suppliers_df["supplier_id"].unique())

missing_suppliers = supplier_ids_products - supplier_ids_master
print("Suppliers missing in supplier master:", len(missing_suppliers))

Suppliers missing in supplier master: 0


## Missing Value Analysis

Missing values can represent:
- Data collection gaps
- Operational issues
- Incorrect joins

We identify and treat them carefully.

In [10]:
def missing_summary(df):
    return df.isna().sum().sort_values(ascending=False)

missing_summary(sales_df)

date                        0
sku_id                      0
units_sold                  0
selling_price               0
gross_revenue               0
promo_flag                  0
discount_pct                0
month                       0
week_of_year                0
day_of_week                 0
is_weekend                  0
is_festival_month           0
is_payday_period            0
season_tag                  0
holiday_flag                0
weather_index               0
campaign_intensity          0
platform_traffic_source     0
traffic_index               0
competitor_price_index      0
competitor_stockout_flag    0
bundle_offer_flag           0
stock_visibility_score      0
rating_score                0
review_volume               0
product_visibility_rank     0
dtype: int64

In [11]:
missing_summary(inventory_df)

sku_id                  0
current_stock           0
safety_stock            0
reorder_point           0
lead_time_days          0
inbound_shipment_qty    0
supplier_id             0
dtype: int64

In [12]:
missing_summary(products_df)

sku_id          0
sku_name        0
category        0
sub_category    0
mrp             0
cost_price      0
supplier_id     0
dtype: int64

In [13]:
missing_summary(suppliers_df)

supplier_id              0
supplier_name            0
avg_lead_time            0
lead_time_variability    0
dtype: int64

## Business Rule Checks

We now validate logical rules based on how inventory systems work.

In [14]:
# Negative sales
negative_sales = sales_df[sales_df["units_sold"] < 0]
print("Negative sales records:", negative_sales.shape[0])

# Negative inventory
negative_inventory = inventory_df[inventory_df["current_stock"] < 0]
print("Negative inventory records:", negative_inventory.shape[0])

# Invalid supplier lead times
invalid_lead_time = suppliers_df[suppliers_df["avg_lead_time"] <= 0]
print("Invalid lead time records:", invalid_lead_time.shape[0])

Negative sales records: 0
Negative inventory records: 0
Invalid lead time records: 0


## Sales with Zero Inventory Check

Sales recorded when on-hand inventory is zero may indicate stockouts,
backorders, or delayed inventory updates.

In [15]:
inventory_zero = inventory_df[inventory_df["current_stock"] == 0][
    ["sku_id"]
]

sales_with_zero_inventory = sales_df.merge(
    inventory_zero,
    left_on=["sku_id"],
    right_on=["sku_id"],
    how="inner"
)

print("Potential stockout-related sales records:", sales_with_zero_inventory.shape[0])

Potential stockout-related sales records: 0


## Data Corrections & Assumptions

Based on the validation checks, the following actions are taken:

- Negative sales values are treated as data errors and removed
- Negative inventory values are clipped to zero
- Missing lead times are filled using supplier-level averages
- Records with unresolved SKU or supplier references are excluded

All assumptions are documented to maintain transparency.

In [16]:
# Remove negative sales
sales_df = sales_df[sales_df["units_sold"] >= 0]

# Clip negative inventory
inventory_df["current_stock"] = inventory_df["current_stock"].clip(lower=0)

# Fill missing lead times
suppliers_df["avg_lead_time"] = suppliers_df["avg_lead_time"].fillna(
    suppliers_df["avg_lead_time"].median()
)

## Save Cleaned Datasets

The cleaned datasets will be used in all downstream notebooks.

In [17]:
from pathlib import Path
BASE_PATH = Path("../")
DATA_PATH_PROCESSED = BASE_PATH / "data" / "processed"

sales_df.to_csv(DATA_PATH_PROCESSED / "sales_clean.csv", index=False)
inventory_df.to_csv(DATA_PATH_PROCESSED / "inventory_clean.csv", index=False)
products_df.to_csv(DATA_PATH_PROCESSED / "products_clean.csv", index=False)
suppliers_df.to_csv(DATA_PATH_PROCESSED / "suppliers_clean.csv", index=False)

print("Cleaned data saved to:", DATA_PATH_PROCESSED)

Cleaned data saved to: ..\data\processed
